In [3]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

In [4]:
load_dotenv()

True

In [5]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [6]:
class ResumeState(TypedDict):
    resume: str

    skill_feedback: str
    experience_feedback: str
    eduction_feedback: str

    final_decision: str

In [7]:
def skill_review(state: ResumeState):
    prompt = f"""
        You are an HR Recruiter
        Analyze ONLY the technical skills in the resume.

        Resume:
        {state["resume"]}

        Give a short evaluation.
    """
    response = llm.invoke(prompt)

    return {
        "skill_feedback": response.content
    }

In [8]:
def experience_review(state: ResumeState):
    prompt = f"""
        You are an HR recruiter.

        Analyze Only the candidate's work experience.

        Resume:
        {state['resume']}

        Give a short evaluation
    """

    response = llm.invoke(prompt)

    return {
        "experience_feedback": response.content
    }

In [9]:
def eduction_review(state: ResumeState):
    prompt = f"""
        You are an HR Recruiter
        Analyze ONLY the candidate's eduction.

        Resume:
        {state["resume"]}

        Give a short evalution.
    """
    response = llm.invoke(prompt)
    return {
        "education_feedback":response.content
    }

In [10]:
builder = StateGraph(ResumeState)

In [11]:
builder.add_node('skill_review', skill_review)
builder.add_node('experience_review', experience_review)
builder.add_node('eduction_review', eduction_review)

In [12]:
builder.add_edge(START, "skill_review")
builder.add_edge(START, "experience_review")
builder.add_edge(START, "education_review")

In [13]:
def final_decison(state: ResumeState):
    prompt = f"""
    You are a Senior Hiring Manager.
    Below are the evaluation from the different reviews.

    skills:
    {state['skill_feedback']}

    Experience:
    {state['experience_feedback']}

    Eduction:
    {state['eduction_feedback']}

    Based on this evaluations, should the candidate move to the next interview round?
    Give a short final decision.
    """

    response = llm.invoke(prompt)
    return {
        "final_decision" : response.content
    }


In [14]:
builder.add_edge("skill_review", "final_decison")
builder.add_edge("experience_review", "final_decision")
builder.add_edge("eduction_review", "final_decison")

In [15]:
builder.add_edge("final_decison", END)